# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Extract the metadata object (not subscriptable)
meta = dataset.metadata
# Pretty-print dataset title and description
print(f"{meta.name}: {meta.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. We will enumerate all record sets and, for each, show its ID, label, and available fields/columns.

In [ ]:
# List all record sets and their fields using their @id
print("Available record sets and associated fields in the dataset:")
record_sets = list(dataset.record_sets)
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print("  Fields:")
    field_ids = []
    for f in rs.fields:
        print(f"    Field @id: {f.id}, Name: {f.name}, Data type: {f.data_type}")
        field_ids.append(f.id)
        # If the field maps to a column, display those
        if hasattr(f, 'column') and f.column:
            column = f.column
            print(f"      Column @id: {column.id if hasattr(column, 'id') else column}, Name: {column.name if hasattr(column, 'name') else column}")
    print()
    record_set_ids.append(rs.id)


## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview.

In [ ]:
# Extract all record sets into a dict of DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"--- Records from {rs_id} ---")
        print(df.head())
        print()
    else:
        print(f"No records found for record set: {rs_id}\n")
# For illustration, we select the main survey data record set if its common name/id is present
# Otherwise, you may explicitly specify your primary record set here.
main_record_set_id = None
for rs in record_sets:
    if 'survey' in rs.name.lower() or 'responses' in rs.name.lower() or 'main' in rs.name.lower():
        main_record_set_id = rs.id
        break
if not main_record_set_id and len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]   # fallback to the first found
# List columns of the main record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in main record set ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No suitable main record set found for demonstration.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Select a numeric field for analysis
# We'll try to use fields like PHQ-9 or GAD-7 scores, commonly found in mental health datasets

df = dataframes.get(main_record_set_id)

numeric_candidates = [col for col in (df.columns if df is not None else []) if any(key in col.lower() for key in ('phq', 'gad', 'psq', 'score', 'age'))]
if len(numeric_candidates) == 0 and df is not None:
    numeric_candidates = df.select_dtypes(include='number').columns.tolist()
if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field for analysis: {numeric_field}")
else:
    print("No suitable numeric field found in main record set.")

if df is not None and numeric_candidates:
    # Example filtering and normalization
    threshold = df[numeric_field].mean()  # Use mean as an example threshold
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    # Group by a relevant field (e.g., 'gender' or 'village' if present)
    group_field = None
    for candidate in ['gender', 'sex', 'village', 'education', 'marital']:
        for col in df.columns:
            if candidate in col.lower():
                group_field = col
                break
        if group_field: break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame(f"mean_{numeric_field}")
        print(f"\nGrouped data by {group_field}, mean {numeric_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found for aggregation.")
else:
    print("Data unavailable or no numeric columns for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_candidates:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, color='skyblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # Boxplot by group if possible
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated step-by-step how to load, overview, extract, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library. We explored available record sets, fields, and conducted basic exploratory data analysis, including filtering and normalizing a key survey score and visualizing its distribution.

Such workflows can be adapted to new datasets with Croissant schemas for seamless reproducible analysis in research and public health.
